## 1) Imports

In [8]:
import cv2
import os
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
import mediapipe as mp
from sklearn.model_selection import train_test_split

## 2) Setup

In [9]:
letters      = [chr(c) for c in range(ord("A"), ord("Z")+1)]
digits       = [str(d) for d in range(10)]
class_names  = letters + digits
class_to_idx = {c: i for i, c in enumerate(class_names)}

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE   = 224
DATA_DIR   = "webcam_data"
OUT_DIR    = "runs_asl36_final"
MODEL_PATH = os.path.join(OUT_DIR, "best_resnet18.pt")

def build_resnet18(num_classes):
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

print("DEVICE:", DEVICE)
print("Classes:", len(class_names))
print("MODEL_PATH exists?", os.path.exists(MODEL_PATH))

DEVICE: cuda
Classes: 36
MODEL_PATH exists? True


## 3) MediaPipe Setup

In [10]:
mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.6
)

def get_hand_crop(frame, hand_landmarks, padding_ratio=0.20, max_frame_fraction=0.60):
    h, w = frame.shape[:2]

    x_coords = [lm.x * w for lm in hand_landmarks.landmark]
    y_coords = [lm.y * h for lm in hand_landmarks.landmark]

    x_min, x_max = int(min(x_coords)), int(max(x_coords))
    y_min, y_max = int(min(y_coords)), int(max(y_coords))

    box_w = x_max - x_min
    box_h = y_max - y_min
    side  = max(box_w, box_h)

    cx = (x_min + x_max) // 2
    cy = (y_min + y_max) // 2

    pad  = int(side * padding_ratio)
    half = side // 2 + pad

    x1 = max(0, cx - half)
    y1 = max(0, cy - half)
    x2 = min(w, cx + half)
    y2 = min(h, cy + half)

    if x2 <= x1 or y2 <= y1:
        return None

    crop_area  = (x2 - x1) * (y2 - y1)
    frame_area = h * w
    if crop_area / frame_area > max_frame_fraction:
        return None

    return frame[y1:y2, x1:x2]

print("MediaPipe initialized.")

MediaPipe initialized.


## 4) Data Collection Function

In [11]:
def collect_webcam_data(label, num_images=200):
    save_dir = os.path.join(DATA_DIR, label)
    os.makedirs(save_dir, exist_ok=True)

    existing = len([f for f in os.listdir(save_dir)
                    if f.lower().endswith((".jpg", ".png"))])

    if existing >= num_images:
        print(f"Already have {existing} images for {label} — skipping.")
        return

    cap   = cv2.VideoCapture(0)
    count = existing
    print(f"\nCollecting {label}: need {num_images - existing} more images")
    print("Show the sign clearly — SPACE to capture, Q to skip to next")

    while count < num_images:
        ret, frame = cap.read()
        if not ret:
            break

        frame     = cv2.flip(frame, 1)
        rgb       = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results   = hands.process(rgb)
        display   = frame.copy()

        hand_visible = False
        if results.multi_hand_landmarks:
            hand_lm = results.multi_hand_landmarks[0]
            mp_drawing.draw_landmarks(
                display, hand_lm, mp_hands.HAND_CONNECTIONS
            )
            crop = get_hand_crop(frame, hand_lm)
            if crop is not None:
                hand_visible = True

        status_color = (0, 255, 0) if hand_visible else (0, 0, 255)
        cv2.putText(display,
                    f"Sign: {label} | {count}/{num_images} | SPACE=save Q=next",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)

        if not hand_visible:
            cv2.putText(display, "NO HAND DETECTED",
                        (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

        cv2.imshow("ASL Data Collection", display)
        key = cv2.waitKey(1) & 0xFF

        if key == ord(" ") and hand_visible:
            path = os.path.join(save_dir, f"{label}_{count}.jpg")
            cv2.imwrite(path, crop)
            count += 1
            print(f"  Saved {count}/{num_images}")
        elif key == ord("q"):
            print(f"  Skipped — saved {count} images for {label}")
            break

    cap.release()
    cv2.destroyAllWindows()
    print(f"Done: {label} — {count} images saved.")

## 5) Collect all 36 signs

In [12]:
all_labels = letters + digits

for label in all_labels:
    collect_webcam_data(label, num_images=200)

print("\nAll data collection complete.")
print("Images saved in:", DATA_DIR)

print("\nSummary:")
for label in all_labels:
    folder = os.path.join(DATA_DIR, label)
    if os.path.exists(folder):
        count = len([f for f in os.listdir(folder)
                     if f.lower().endswith((".jpg", ".png"))])
        print(f"  {label}: {count} images")

Already have 200 images for A — skipping.
Already have 200 images for B — skipping.
Already have 200 images for C — skipping.
Already have 200 images for D — skipping.
Already have 200 images for E — skipping.
Already have 200 images for F — skipping.
Already have 200 images for G — skipping.
Already have 200 images for H — skipping.
Already have 200 images for I — skipping.
Already have 200 images for J — skipping.
Already have 200 images for K — skipping.
Already have 200 images for L — skipping.
Already have 200 images for M — skipping.
Already have 200 images for N — skipping.
Already have 200 images for O — skipping.
Already have 200 images for P — skipping.
Already have 200 images for Q — skipping.
Already have 200 images for R — skipping.
Already have 200 images for S — skipping.
Already have 200 images for T — skipping.
Already have 200 images for U — skipping.
Already have 200 images for V — skipping.
Already have 200 images for W — skipping.
Already have 200 images for X — sk

## 6) Dataset Class

In [13]:
finetune_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

all_webcam_samples = []
for label in os.listdir(DATA_DIR):
    folder = os.path.join(DATA_DIR, label)
    if not os.path.isdir(folder) or label not in class_to_idx:
        continue
    for fn in os.listdir(folder):
        if fn.lower().endswith((".jpg", ".png")):
            all_webcam_samples.append(
                (os.path.join(folder, fn), class_to_idx[label])
            )

labels_only = [y for _, y in all_webcam_samples]
train_samples_ft, val_samples_ft = train_test_split(
    all_webcam_samples, test_size=0.15, stratify=labels_only, random_state=42
)
print(f"Fine-tune train: {len(train_samples_ft)}, val: {len(val_samples_ft)}")

class SimpleDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, y = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), y

train_loader_ft = DataLoader(
    SimpleDataset(train_samples_ft, finetune_tf),
    batch_size=32, shuffle=True, num_workers=0
)
val_loader_ft = DataLoader(
    SimpleDataset(val_samples_ft, val_tf),
    batch_size=64, shuffle=False, num_workers=0
)

Fine-tune train: 6120, val: 1080


## 7) Finetune ResNet18

In [14]:
model = build_resnet18(len(class_names))
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))

# Swap head to add dropout before fine-tuning
model.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(512, len(class_names))
)
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scaler    = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

PATIENCE   = 3
best_val   = -1.0
no_improve = 0
ft_path    = os.path.join(OUT_DIR, "best_resnet18_finetuned.pt")

print("Fine-tuning ResNet18 on webcam data...")
print(f"Train: {len(train_samples_ft)} | Val: {len(val_samples_ft)}")

for epoch in range(15):
    model.train()
    losses, correct, total = [], 0, 0

    for xb, yb in train_loader_ft:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
            logits = model(xb)
            loss   = criterion(logits, yb)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        losses.append(loss.item())
        correct += (logits.argmax(1) == yb).sum().item()
        total   += yb.size(0)

    train_acc = correct / total

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for xb, yb in val_loader_ft:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
                logits = model(xb)
            val_correct += (logits.argmax(1) == yb).sum().item()
            val_total   += yb.size(0)

    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1:02d}/15 | "
          f"train_loss={sum(losses)/len(losses):.4f} | "
          f"train_acc={train_acc:.4f} | "
          f"val_acc={val_acc:.4f}")

    if val_acc > best_val:
        best_val   = val_acc
        no_improve = 0
        torch.save(model.state_dict(), ft_path)
        print(f"  New best saved ({best_val:.4f})")
    else:
        no_improve += 1
        print(f"  No improvement ({no_improve}/{PATIENCE})")
        if no_improve >= PATIENCE:
            print("Early stopping triggered.")
            break

model.load_state_dict(torch.load(ft_path, map_location=DEVICE, weights_only=True))
model.eval()
print(f"\nBest val acc: {best_val:.4f}")
print(f"Fine-tuned model saved to: {ft_path}")

Fine-tuning ResNet18 on webcam data...
Train: 6120 | Val: 1080
Epoch 01/15 | train_loss=2.3968 | train_acc=0.4686 | val_acc=0.9667
  New best saved (0.9667)
Epoch 02/15 | train_loss=0.7922 | train_acc=0.9507 | val_acc=0.9935
  New best saved (0.9935)
Epoch 03/15 | train_loss=0.3335 | train_acc=0.9806 | val_acc=0.9926
  No improvement (1/3)
Epoch 04/15 | train_loss=0.1815 | train_acc=0.9882 | val_acc=0.9981
  New best saved (0.9981)
Epoch 05/15 | train_loss=0.1174 | train_acc=0.9940 | val_acc=0.9981
  No improvement (1/3)
Epoch 06/15 | train_loss=0.0777 | train_acc=0.9972 | val_acc=0.9991
  New best saved (0.9991)
Epoch 07/15 | train_loss=0.0554 | train_acc=0.9979 | val_acc=0.9991
  No improvement (1/3)
Epoch 08/15 | train_loss=0.0419 | train_acc=0.9990 | val_acc=0.9991
  No improvement (2/3)
Epoch 09/15 | train_loss=0.0331 | train_acc=0.9993 | val_acc=1.0000
  New best saved (1.0000)
Epoch 10/15 | train_loss=0.0265 | train_acc=0.9993 | val_acc=1.0000
  No improvement (1/3)
Epoch 11/15 